# E4 — cabeza de equipo neuronal (entrenamiento en Colab)

Entrena `src/owner_nn.py` con el banco **DFL** exportado (CC BY 4.0): 7 modelos dejando un partido fuera + 1 final con los 7, **cada uno con 3 semillas** (ensemble E5; se promedian los log-odds).

**Reglas:** aquí entra **solo** `colab_bundle.zip` (banco DFL + código). Nada del reto: ni ATD, ni vídeo, ni Smart Tagging, ni homografía. **Gemini desactivado** en este notebook.

1. *Entorno de ejecución → Cambiar tipo → GPU (A100 si la tienes; T4 también sirve, más lento).*
2. Sube `external/nn/colab_bundle.zip` a `MyDrive/e4/` (o súbelo en la celda 2).
3. Ejecuta las celdas en orden. La celda 4 se puede relanzar si Colab se desconecta: retoma por fold y semilla.
4. Descarga `oof_rnn.npz`, `owner_nn_s0.pt`, `owner_nn_s1.pt`, `owner_nn_s2.pt` y `owner_nn.json` y déjalos en `external/nn/out/` en local.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/e4'   # los resultados quedan en tu Drive: sobreviven a una desconexión
import os; os.makedirs(WORK, exist_ok=True); print(os.listdir(WORK))

In [ ]:
# Solo si el zip no está ya en MyDrive/e4
import os, shutil
if not os.path.exists(f'{WORK}/colab_bundle.zip'):
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), f'{WORK}/colab_bundle.zip')
!rm -rf /content/e4 && mkdir -p /content/e4 && unzip -q {WORK}/colab_bundle.zip -d /content/e4 && ls -la /content/e4 /content/e4/src

In [ ]:
# Smoke test (~1-2 min): simetría exacta del modelo + un fold diminuto de punta a punta
%cd /content/e4
!python -m pytest -q tests/test_nn_inputs.py
!python -m src.owner_nn --data nn_bench.npz --out /content/smoke --smoke

In [ ]:
# Entrenamiento real: 7 folds + modelo final, 3 semillas cada uno (24 entrenamientos). Reanudable: salta lo ya escrito en WORK/out
%cd /content/e4
!python -m src.owner_nn --data nn_bench.npz --out {WORK}/out 2>&1 | tee -a {WORK}/out_log.txt

In [ ]:
from google.colab import files
for f in ('oof_rnn.npz', 'owner_nn_s0.pt', 'owner_nn_s1.pt', 'owner_nn_s2.pt', 'owner_nn.json'):
    files.download(f'{WORK}/out/{f}')
files.download(f'{WORK}/out_log.txt')